<a href="https://colab.research.google.com/github/zhangling297/deep-learning-with-python-notebooks/blob/master/CS599_Homework_2_Text_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part 1: Set Up

In [ ]:
# Create a Text Classificationwith a Feed Foward Neural Network to achieve 90% plus accuracy on the test sets of  text classification tasks of Glue and Superglue datasets as given below https://huggingface.co/datasets/aps/super_glue/viewer/rte and https://huggingface.co/datasets/nyu-mll/glue/viewer/sst2Links to an external site.

!pip install -q datasets

import pandas as pd
from datasets import load_dataset
import random
import torch
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm

# enable tqdm in pandas
tqdm.pandas()

# Check the gpu
use_gpu = True

# Select device
device = torch.device("cuda" if use_gpu and torch.cuda.is_available() else "cpu")
print(f'device: {device.type}')







# Data Load

In [ ]:


# Load GLUE SST-2 dataset
print("Loading GLUE SST-2 dataset...")
glue_sst2 = load_dataset("glue", "sst2")
train_df_glue = glue_sst2['train'].to_pandas()
train_df_glue.columns = ['sentence', 'label', 'idx']
display(train_df_glue.head())

# Load SuperGLUE RTE dataset
print("\nLoading SuperGLUE RTE dataset...")
superglue_rte = load_dataset("super_glue", "rte")
train_df_superglue = superglue_rte['train'].to_pandas()
train_df_superglue.columns = ['premise', 'hypothesis', 'label', 'idx']

display(train_df_superglue.head())


# Feed Foward Neuro Network implementation.

*   **Softmax forward pass**
*   **Backpropergation**
*   **Gradient Descent**





In [ ]:
# Data Preprocessing
# Concatenate premise and hypothesis for SuperGLUE
train_df_superglue['text'] = train_df_superglue['premise'] + " [SEP] " + train_df_superglue['hypothesis']
df_SuperGlue = train_df_superglue[['text', 'label', 'idx']].copy()

# Standardize the GLUE dataframe to have a 'text' column
df_Glue = train_df_glue.rename(columns={'sentence': 'text'})

print("SuperGLUE Prepared Data:")
display(df_SuperGlue.head())


# Implement softmax manually ---
def softmax(z):
    # Subtract max for numerical stability
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

class ManualFFNN:
    def __init__(self, input_dim, hidden_dim, output_dim, learning_rate=0.01):
        # Initialize weights and biases
        self.W1 = np.random.randn(input_dim, hidden_dim) * 0.01
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.b2 = np.zeros((1, output_dim))
        self.lr = learning_rate

    def forward(self, X):
        # Layer 1 (Hidden layer with ReLU activation)
        self.Z1 = np.dot(X, self.W1) + self.b1
        self.A1 = np.maximum(0, self.Z1)  # ReLU

        # Layer 2 (Output layer with Softmax)
        self.Z2 = np.dot(self.A1, self.W2) + self.b2
        self.probs = softmax(self.Z2)
        return self.probs

    # --- 2. Implement backprop manually ---
    def backward(self, X, y_true_one_hot):
        m = X.shape[0]  # Number of examples

        # Output layer error (Gradient of cross-entropy loss with softmax)
        dZ2 = self.probs - y_true_one_hot

        # Gradients for Layer 2
        dW2 = np.dot(self.A1.T, dZ2) / m
        db2 = np.sum(dZ2, axis=0, keepdims=True) / m

        # Hidden layer error (Backprop through ReLU)
        dA1 = np.dot(dZ2, self.W2.T)
        dZ1 = dA1 * (self.Z1 > 0)  # Derivative of ReLU

        # Gradients for Layer 1
        dW1 = np.dot(X.T, dZ1) / m
        db1 = np.sum(dZ1, axis=0, keepdims=True) / m

        # Call Gradient Descent step
        self.update_weights(dW1, db1, dW2, db2)

    # --- 1. Implement gradient descent manually ---
    def update_weights(self, dW1, db1, dW2, db2):
        # Update weights and biases using the calculated gradients
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2


# Run Experiments Model Comparision Using GLUE Dataset

**Model Accuracy & F1 Convergence Speed**


1. *Model Training Accuracy -Highest*:

*   Voted Perceptron: 0.833
*   Logistic Regression: 0.817
*   Feed Foreard Neuro Network: 0.567 than flat



2. *F1 Convergence - Highest*:


*   Voted Perceptron: 0.86
*   Logistic Regression: 0.85
*   FFNN: 0.567





In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import numpy as np

# 1. Vectorize Data (Using GLUE SST-2 for this experiment)
print("Vectorizing data...")
vectorizer = CountVectorizer(max_features=1000)  # Starting with 1k vocabulary size
# Using a subset of data for faster demonstration
subset_df = df_Glue.dropna().sample(n=5000, random_state=42)
X = vectorizer.fit_transform(subset_df['text']).toarray()
y = subset_df['label'].values

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 2. Implement Voted Perceptron
class VotedPerceptron:
    def __init__(self, epochs=15):
        self.epochs = epochs
        self.V = [] # Stores (weight_vector, vote_count)

    def fit(self, X, y):
        w = np.zeros(X.shape[1])
        c = 0
        # Convert 0/1 labels to -1/1 for Perceptron
        y_p = np.where(y == 0, -1, 1)

        history = {'accuracy': [], 'f1': []}

        for epoch in range(self.epochs):
            for i in range(X.shape[0]):
                if y_p[i] * np.dot(w, X[i]) <= 0:
                    self.V.append((w.copy(), c))
                    w = w + y_p[i] * X[i]
                    c = 1
                else:
                    c += 1
            self.V.append((w.copy(), c))

            # Evaluate at the end of each epoch for convergence tracking
            y_pred = self.predict(X)
            history['accuracy'].append(accuracy_score(y, y_pred))
            history['f1'].append(f1_score(y, y_pred))
        return history

    def predict(self, X):
        predictions = []
        for x in X:
            s = sum(c * np.sign(np.dot(w, x)) for w, c in self.V)
            # Convert back to 0/1
            predictions.append(1 if s >= 0 else 0)
        return np.array(predictions)

# 3. Setup comparison loop
epochs = 15
print("Training models...")

# Voted Perceptron
vp = VotedPerceptron(epochs=epochs)
vp_history = vp.fit(X_train, y_train)

# Logistic Regression (simulating epochs by increasing max_iter)
lr_history = {'accuracy': [], 'f1': []}
for e in range(1, epochs + 1):
    lr = LogisticRegression(max_iter=e, solver='saga', warm_start=True, random_state=42)
    lr.fit(X_train, y_train)
    preds = lr.predict(X_train)
    lr_history['accuracy'].append(accuracy_score(y_train, preds))
    lr_history['f1'].append(f1_score(y_train, preds))

# Manual Forward Neural Network
nn = ManualFFNN(input_dim=X_train.shape[1], hidden_dim=64, output_dim=2, learning_rate=0.5)
nn_history = {'accuracy': [], 'f1': []}

# Convert y_train to one-hot encoding for the neural network
y_train_one_hot = np.zeros((y_train.size, 2))
y_train_one_hot[np.arange(y_train.size), y_train] = 1

for epoch in range(epochs):
    # Forward and Backward passes
    probs = nn.forward(X_train)
    nn.backward(X_train, y_train_one_hot)

    # Get predictions and track metrics
    preds = np.argmax(probs, axis=1)
    nn_history['accuracy'].append(accuracy_score(y_train, preds))
    nn_history['f1'].append(f1_score(y_train, preds))

# Plotting Convergence
plt.figure(figsize=(14, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, epochs + 1), vp_history['accuracy'], label='Voted Perceptron', marker='o')
plt.plot(range(1, epochs + 1), lr_history['accuracy'], label='Logistic Regression', marker='s')
plt.plot(range(1, epochs + 1), nn_history['accuracy'], label='Manual FFNN', marker='^')
plt.title('Training Accuracy Convergence')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(range(1, epochs + 1), vp_history['f1'], label='Voted Perceptron', marker='o')
plt.plot(range(1, epochs + 1), lr_history['f1'], label='Logistic Regression', marker='s')
plt.plot(range(1, epochs + 1), nn_history['f1'], label='Manual FFNN', marker='^')
plt.title('Training F1 Score Convergence')
plt.xlabel('Epochs')
plt.ylabel('F1 Score')
plt.legend()
plt.show()


In [ ]:
readme_content = """=====================================================================
README: Text Classification Experiments (GLUE & SuperGLUE)
=====================================================================

OVERVIEW
This notebook contains a comprehensive pipeline for text classification. It compares the performance of a custom-built Feed Forward Neural Network (FFNN), Logistic Regression, and Voted Perceptron on GLUE (SST-2) and SuperGLUE (BoolQ, RTE) datasets.

REQUIREMENTS
- Google Colab environment (Highly Recommended)
- Python 3.x
- Required Libraries: datasets, torch, scikit-learn, pandas, numpy, matplotlib, tqdm

INSTRUCTIONS TO RUN
1. Setup Environment:
   - Open the notebook in Google Colab.
   - Enable GPU for faster PyTorch training: Go to the top menu -> "Runtime" -> "Change runtime type" -> Select "T4 GPU" -> Click "Save".

2. Install Dependencies:
   - Run the very first setup cell containing `!pip install -q datasets` to install the Hugging Face datasets library.

3. Execution Order:
   - Run the cells sequentially from top to bottom.
   - The notebook is structured into logical experiments:
     * Parts 1 & 2: Environment setup and Data Loading.
     * Part 3: Manual FFNN Implementation (Softmax, Backpropagation, Gradient Descent).
     * Part 4: Model Comparison (Accuracy & Convergence across models).
     * Part 5: Feature Size Sensitivity (TF-IDF at 1k, 5k, 10k features), Depth vs. Performance, and Failure Analysis.
     * Part 6: Optimization Techniques (Regularization, Activations, Learning Rate Tuning, Cost Functions).

IMPORTANT COMMENTS & MODIFICATIONS
- Data Subsampling: For demonstration and speed, some experiments use a subsample of 5,000 - 7,000 rows. You can modify the `max_samples` variables in the code if you wish to train on the entire dataset.
- Feature Engineering: We utilize `TfidfVectorizer` instead of basic Bag-of-Words in the deeper experiments. This scaling prevents gradients from exploding during mini-batch gradient descent.
- PyTorch vs. Manual: The manual FFNN serves as an educational foundation. The PyTorch FFNN implementation utilizes optimized Adam solvers and Dropout, showcasing how modern frameworks achieve superior high-90s% training accuracy.
====================================================================="""

# Write to README.txt
with open("README.txt", "w") as f:
    f.write(readme_content)

print("README.txt has been successfully generated in your working directory!\n")
print("--- FILE CONTENTS ---\n")
print(readme_content)


# Model Comparision Using SuperGLUE Dataset

**Model Accuracy & F1 Convergence Speed**


 **Accuracy comparison**:

*   Voted Perceptron 0.977 (Training) & 0.655 (Test)
*   Logistic Regression 0.933 (Training) & 0.652 (Test)
*   FFNN 0.932 (Training) & 0.527 (Test)

**F1 scores by model**:

*   Voted Perceptron 0.982(Train) & 0.722 (Test)
*   Logiistic regression 0.948 (Train) & 0.741 (Test)
*   FFNN 0.713 (Train) & 0.472 (Test)

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle

# =========================================================
# 1. Load SuperGLUE BoolQ
# =========================================================
print("Loading SuperGLUE BoolQ...")
ds = load_dataset("super_glue", "boolq")
train_data = ds["train"]
texts = [f"question: {q} passage: {p}" for q, p in zip(train_data["question"], train_data["passage"])]
labels = np.array(train_data["label"], dtype=np.int64)

max_samples = 7000
if len(texts) > max_samples:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(texts), size=max_samples, replace=False)
    texts = [texts[i] for i in idx]
    labels = labels[idx]

X_train_text, X_test_text, y_train, y_test = train_test_split(texts, labels, test_size=0.2, random_state=42, stratify=labels)

# =========================================================
# 2. Vectorize text (Optimal: 5000 features)
# =========================================================
print("Vectorizing data...")
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), lowercase=True, stop_words="english", sublinear_tf=True)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)

# =========================================================
# 3. Logistic Regression (Baseline)
# =========================================================
lr = LogisticRegression(max_iter=100, solver="saga", penalty="l2", C=5.0, random_state=42)
lr.fit(X_train, y_train)
lr_train_pred = lr.predict(X_train)
lr_test_pred = lr.predict(X_test)

# =========================================================
# 4. Manual FFNN (Optimal Params)
# =========================================================
class ManualFFNN:
    def __init__(self, input_dim, hidden1=64, hidden2=32, output_dim=2, learning_rate=0.1, seed=42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, np.sqrt(2 / input_dim), size=(input_dim, hidden1)).astype(np.float32)
        self.b1 = np.zeros((1, hidden1), dtype=np.float32)
        self.W2 = rng.normal(0, np.sqrt(2 / hidden1), size=(hidden1, hidden2)).astype(np.float32)
        self.b2 = np.zeros((1, hidden2), dtype=np.float32)
        self.W3 = rng.normal(0, np.sqrt(2 / hidden2), size=(hidden2, output_dim)).astype(np.float32)
        self.b3 = np.zeros((1, output_dim), dtype=np.float32)
        self.lr = learning_rate

    def relu(self, z): return np.maximum(0, z)
    def relu_grad(self, z): return (z > 0).astype(np.float32)
    def softmax(self, z):
        z_shifted = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z_shifted)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = self.relu(self.Z1)
        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self.relu(self.Z2)
        self.Z3 = self.A2 @ self.W3 + self.b3
        self.probs = self.softmax(self.Z3)
        return self.probs

    def backward(self, X, y_one_hot):
        m = X.shape[0]
        dZ3 = (self.probs - y_one_hot) / m
        dW3 = self.A2.T @ dZ3
        db3 = np.sum(dZ3, axis=0, keepdims=True)
        dA2 = dZ3 @ self.W3.T
        dZ2 = dA2 * self.relu_grad(self.Z2)
        dW2 = self.A1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)
        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_grad(self.Z1)
        dW1 = X.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def predict(self, X): return np.argmax(self.forward(X), axis=1)

    def fit(self, X, y, epochs=30, batch_size=32):
        y_one_hot = np.zeros((len(y), 2), dtype=np.float32)
        y_one_hot[np.arange(len(y)), y] = 1.0
        n = X.shape[0]
        for epoch in range(epochs):
            X_shuf, y_shuf_oh = shuffle(X, y_one_hot, random_state=epoch)
            for start in range(0, n, batch_size):
                end = start + batch_size
                self.forward(X_shuf[start:end])
                self.backward(X_shuf[start:end], y_shuf_oh[start:end])

manual_nn = ManualFFNN(input_dim=X_train.shape[1], hidden1=64, hidden2=32, output_dim=2, learning_rate=0.1, seed=42)
print("Training Manual FFNN...")
manual_nn.fit(X_train, y_train, epochs=30, batch_size=32)
nn_train_pred = manual_nn.predict(X_train)
nn_test_pred = manual_nn.predict(X_test)

# =========================================================
# 5. PyTorch FFNN
# =========================================================
class TorchFFNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )
    def forward(self, x):
        return self.net(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch_model = TorchFFNN(X_train.shape[1]).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(torch_model.parameters(), lr=0.001, weight_decay=1e-4) # Adam + L2

train_dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train))
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

print("Training PyTorch FFNN...")
torch_model.train()
for epoch in range(15):
    for batch_X, batch_y in train_loader:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = torch_model(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

torch_model.eval()
with torch.no_grad():
    pt_train_pred = torch.argmax(torch_model(torch.tensor(X_train).to(device)), axis=1).cpu().numpy()
    pt_test_pred = torch.argmax(torch_model(torch.tensor(X_test).to(device)), axis=1).cpu().numpy()

# =========================================================
# 6. Final Results Comparison
# =========================================================
print("\n--- Final Results (Optimized Parameters) ---")
print(f"Logistic Regression  -> Train Acc: {accuracy_score(y_train, lr_train_pred):.4f} | Test Acc: {accuracy_score(y_test, lr_test_pred):.4f}")
print(f"Manual FFNN          -> Train Acc: {accuracy_score(y_train, nn_train_pred):.4f} | Test Acc: {accuracy_score(y_test, nn_test_pred):.4f}")
print(f"PyTorch FFNN (Adam)  -> Train Acc: {accuracy_score(y_train, pt_train_pred):.4f} | Test Acc: {accuracy_score(y_test, pt_test_pred):.4f}")




# Feature Size Sensitivity Comparision Using GLUE (SuperGLUE has higher overall accuracy)

**Vocabulary sizes: 1k, 5k, 10k**

Train accuracy: about 0.7605 for 0.98 Senconds (1k), 0.7390 for 3.35 Secnonds (5k), and o.8895 for 5.91 seconds (10k);
Test accuracy: about 0.7220 (1k), 0.6320 (5k), and 0.7220 (10k) for 5.91 seconds.

**Overfitting**: As the vocabulary size increases (especially at 10,000 features), the gap between Training Accuracy (88.95%) and Test Accuracy (72.20%) grows significantly. This is a classic sign of overfitting; the model is memorizing rare words that appear in the training set but don't help it generalize to the test set.
Training Time: The training time clearly scales with the vocabulary size (from ~1 second up to ~5.4 seconds). More features mean larger weight matrices, which require more calculations during the forward and backward passes



In [ ]:
import time
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.utils import shuffle
import matplotlib.pyplot as plt
import numpy as np

# Prepare tracking structures
vocab_sizes = [1000, 5000, 10000]
results = {'vocab_size': [], 'train_acc': [], 'test_acc': [], 'train_time': []}

# We'll use a subset of GLUE SST-2 for this experiment
subset_df = df_Glue.dropna().sample(n=5000, random_state=42)
y = subset_df['label'].values
y_one_hot = np.zeros((y.size, 2))
y_one_hot[np.arange(y.size), y] = 1

print("Starting Feature Size Sensitivity Experiment (Mini-batch)...")

for size in vocab_sizes:
    print(f"\nProcessing Vocabulary Size: {size}...")

    # 1. Vectorize Data (Using TF-IDF for stable gradients with mini-batches)
    vectorizer = TfidfVectorizer(max_features=size)
    X = vectorizer.fit_transform(subset_df['text']).toarray().astype(np.float32)
    actual_size = X.shape[1]

    # 2. Train-Test Split
    X_train, X_test, y_train, y_test, y_train_oh, y_test_oh = train_test_split(
        X, y, y_one_hot, test_size=0.2, random_state=42
    )

    # 3. Initialize Model (Using the updated parameters: hidden1 and hidden2)
    nn = ManualFFNN(input_dim=actual_size, hidden1=64, hidden2=32, output_dim=2, learning_rate=0.5, seed=42)

    # 4. Train Model with Mini-Batches and measure time
    start_time = time.time()
    epochs = 15
    batch_size = 64
    n_samples = X_train.shape[0]

    for epoch in range(epochs):
        # Shuffle data at the start of each epoch
        X_shuf, y_shuf_oh = shuffle(X_train, y_train_oh, random_state=epoch)

        for start in range(0, n_samples, batch_size):
            end = start + batch_size
            X_batch = X_shuf[start:end]
            y_batch = y_shuf_oh[start:end]

            nn.forward(X_batch)
            nn.backward(X_batch, y_batch)

    end_time = time.time()

    # 5. Evaluate for Overfitting (Train vs Test Accuracy)
    train_probs = nn.forward(X_train)
    train_preds = np.argmax(train_probs, axis=1)
    train_acc = accuracy_score(y_train, train_preds)

    test_probs = nn.forward(X_test)
    test_preds = np.argmax(test_probs, axis=1)
    test_acc = accuracy_score(y_test, test_preds)

    # 6. Record Results
    results['vocab_size'].append(size)
    results['train_acc'].append(train_acc)
    results['test_acc'].append(test_acc)
    results['train_time'].append(end_time - start_time)

    print(f"Completed {size} (Actual: {actual_size}) | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Time: {end_time - start_time:.2f}s")

# Plotting Overfitting & Training Time
fig, ax1 = plt.subplots(figsize=(10, 6))

# Accuracy Plot (Left Y-Axis)
ax1.set_xlabel('Requested Vocabulary Size')
ax1.set_ylabel('Accuracy', color='tab:blue')
ax1.plot(results['vocab_size'], results['train_acc'], marker='o', label='Train Accuracy', color='tab:blue')
ax1.plot(results['vocab_size'], results['test_acc'], marker='s', label='Test Accuracy', color='tab:cyan', linestyle='--')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Time Plot (Right Y-Axis)
ax2 = ax1.twinx()
ax2.set_ylabel('Training Time (Seconds)', color='tab:red')
ax2.plot(results['vocab_size'], results['train_time'], marker='^', label='Training Time', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax2.legend(loc='lower right')

plt.title('Feature Size Sensitivity: Overfitting and Training Time vs Vocabulary Size')
plt.xticks(vocab_sizes)
plt.show()


# Feature Size Sensitivity Comparision Using SuperGLUE Dataset
**Observe and report Overfitting and Training time**
The "Sweet Spot": The test accuracy actually peaked at 5,000 features (66.21%). This means 5,000 words provided enough information for the model to generalize well without introducing too much noise.
**Overfitting at 10k Features**: When increasing the vocabulary to 10,000 features, the training accuracy jumped up to 93.50%, but the test accuracy dropped to 60.86%. This is a classical example of overfitting—the model started memorizing rare words in the training set that didn't help it predict new examples.
**Training Time Trade-off**: The computational cost scaled dramatically. Training on 1,000 features took only ~1.9 seconds, while 10,000 features took over 15 seconds.
Conclusion: More features are not always better. For this dataset, 5,000 features offers the best balance of generalization and computational efficiency.

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt

from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# =========================================================
# 1. Load SuperGLUE BoolQ
# =========================================================
print("Loading SuperGLUE BoolQ...")
ds = load_dataset("super_glue", "boolq")

# Build text from question + passage
texts = [
    f"question: {q} passage: {p}"
    for q, p in zip(ds["train"]["question"], ds["train"]["passage"])
]
y = np.array(ds["train"]["label"], dtype=np.int64)

# Optional subset for faster runs
max_samples = 7000
if len(texts) > max_samples:
    rng = np.random.default_rng(42)
    idx = rng.choice(len(texts), size=max_samples, replace=False)
    texts = [texts[i] for i in idx]
    y = y[idx]

# One-hot labels
y_one_hot = np.zeros((y.size, 2), dtype=np.float32)
y_one_hot[np.arange(y.size), y] = 1.0

# =========================================================
# 2. Prepare tracking structures
# =========================================================
vocab_sizes = [1000, 5000, 10000]
results = {'vocab_size': [], 'train_acc': [], 'test_acc': [], 'train_time': []}

print("Starting Feature Size Sensitivity Experiment on SuperGLUE BoolQ...")

# =========================================================
# 3. Improved manual FFNN
# =========================================================
class ManualFFNN:
    def __init__(self, input_dim, hidden1=64, hidden2=32, output_dim=2, learning_rate=0.1, seed=42):
        rng = np.random.default_rng(seed)

        # He initialization
        self.W1 = rng.normal(0, np.sqrt(2 / input_dim), size=(input_dim, hidden1)).astype(np.float32)
        self.b1 = np.zeros((1, hidden1), dtype=np.float32)

        self.W2 = rng.normal(0, np.sqrt(2 / hidden1), size=(hidden1, hidden2)).astype(np.float32)
        self.b2 = np.zeros((1, hidden2), dtype=np.float32)

        self.W3 = rng.normal(0, np.sqrt(2 / hidden2), size=(hidden2, output_dim)).astype(np.float32)
        self.b3 = np.zeros((1, output_dim), dtype=np.float32)

        self.lr = learning_rate

    def relu(self, z):
        return np.maximum(0, z)

    def relu_grad(self, z):
        return (z > 0).astype(np.float32)

    def softmax(self, z):
        z = z - np.max(z, axis=1, keepdims=True)
        exp_z = np.exp(z)
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def forward(self, X):
        self.Z1 = X @ self.W1 + self.b1
        self.A1 = self.relu(self.Z1)

        self.Z2 = self.A1 @ self.W2 + self.b2
        self.A2 = self.relu(self.Z2)

        self.Z3 = self.A2 @ self.W3 + self.b3
        self.probs = self.softmax(self.Z3)
        return self.probs

    def backward(self, X, y_one_hot):
        m = X.shape[0]

        dZ3 = (self.probs - y_one_hot) / m
        dW3 = self.A2.T @ dZ3
        db3 = np.sum(dZ3, axis=0, keepdims=True)

        dA2 = dZ3 @ self.W3.T
        dZ2 = dA2 * self.relu_grad(self.Z2)
        dW2 = self.A1.T @ dZ2
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = dZ2 @ self.W2.T
        dZ1 = dA1 * self.relu_grad(self.Z1)
        dW1 = X.T @ dZ1
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        self.W3 -= self.lr * dW3
        self.b3 -= self.lr * db3
        self.W2 -= self.lr * dW2
        self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1
        self.b1 -= self.lr * db1

    def predict(self, X):
        probs = self.forward(X)
        return np.argmax(probs, axis=1)

# =========================================================
# 4. Run vocabulary-size experiment
# =========================================================
for size in vocab_sizes:
    print(f"\nProcessing Vocabulary Size: {size}...")

    # Vectorize with TF-IDF instead of raw counts
    vectorizer = TfidfVectorizer(
        max_features=size,
        ngram_range=(1, 2),
        lowercase=True,
        stop_words="english",
        sublinear_tf=True
    )
    X = vectorizer.fit_transform(texts).toarray().astype(np.float32)
    actual_size = X.shape[1]

    # Train-test split
    X_train, X_test, y_train, y_test, y_train_oh, y_test_oh = train_test_split(
        X, y, y_one_hot, test_size=0.2, random_state=42, stratify=y
    )

    # Initialize improved model with optimized hyperparameters
    nn = ManualFFNN(
        input_dim=actual_size,
        hidden1=64,   # Reduced complexity
        hidden2=32,
        output_dim=2,
        learning_rate=0.1, # More stable learning rate
        seed=42
    )

    # Train with mini-batches
    start_time = time.time()
    epochs = 30       # Increased epochs
    batch_size = 32   # Smaller batch size for more frequent updates
    n = X_train.shape[0]

    for epoch in range(epochs):
        rng = np.random.default_rng(epoch)
        indices = np.arange(n)
        rng.shuffle(indices)

        X_train_shuf = X_train[indices]
        y_train_oh_shuf = y_train_oh[indices]

        for start in range(0, n, batch_size):
            end = start + batch_size
            X_batch = X_train_shuf[start:end]
            y_batch = y_train_oh_shuf[start:end]

            nn.forward(X_batch)
            nn.backward(X_batch, y_batch)

    end_time = time.time()

    # Evaluate
    train_preds = nn.predict(X_train)
    train_acc = accuracy_score(y_train, train_preds)

    test_preds = nn.predict(X_test)
    test_acc = accuracy_score(y_test, test_preds)

    # Record results
    results['vocab_size'].append(size)
    results['train_acc'].append(train_acc)
    results['test_acc'].append(test_acc)
    results['train_time'].append(end_time - start_time)

    print(f"Completed {size} (Actual: {actual_size}) | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Time: {end_time - start_time:.2f}s")

# =========================================================
# 5. Plot results
# =========================================================
fig, ax1 = plt.subplots(figsize=(10, 6))

ax1.set_xlabel('Requested Vocabulary Size')
ax1.set_ylabel('Accuracy', color='tab:blue')
ax1.plot(results['vocab_size'], results['train_acc'], marker='o', label='Train Accuracy', color='tab:blue')
ax1.plot(results['vocab_size'], results['test_acc'], marker='s', label='Test Accuracy', color='tab:cyan', linestyle='--')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.set_ylabel('Training Time (Seconds)', color='tab:red')
ax2.plot(results['vocab_size'], results['train_time'], marker='^', label='Training Time', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')
ax2.legend(loc='lower right')

plt.title('Feature Size Sensitivity on SuperGLUE BoolQ: Accuracy and Training Time')
plt.xticks(vocab_sizes)
plt.show()


# Depth VS Performance

In [ ]:
from sklearn.neural_network import MLPClassifier
import time
import matplotlib.pyplot as plt

# Define the architectures: 1 layer, 2 layers, and 4 layers
# We will use 64 neurons per hidden layer for consistency
depths = {
    '1 Layer': (64,),
    '2 Layers': (64, 64),
    '4 Layers': (64, 64, 64, 64)
}

depth_results = {'depth': [], 'train_acc': [], 'test_acc': [], 'train_time': []}

print("Starting Depth vs Performance Experiment...")

for name, hidden_sizes in depths.items():
    print(f"Training {name} model...")

    # Using MLPClassifier to easily manage multiple layers
    mlp = MLPClassifier(hidden_layer_sizes=hidden_sizes, activation='relu',
                        max_iter=50, random_state=42)

    start_time = time.time()
    mlp.fit(X_train, y_train)
    end_time = time.time()

    # Evaluate accuracy
    train_acc = mlp.score(X_train, y_train)
    test_acc = mlp.score(X_test, y_test)

    # Record metrics
    depth_results['depth'].append(name)
    depth_results['train_acc'].append(train_acc)
    depth_results['test_acc'].append(test_acc)
    depth_results['train_time'].append(end_time - start_time)

    print(f"Completed {name} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f} | Time: {end_time - start_time:.2f}s")

# Plotting Depth vs Performance
fig, ax1 = plt.subplots(figsize=(10, 6))

# Accuracy Plot (Left Y-Axis)
ax1.set_xlabel('Network Depth')
ax1.set_ylabel('Accuracy', color='tab:blue')
ax1.plot(depth_results['depth'], depth_results['train_acc'], marker='o', label='Train Accuracy', color='tab:blue')
ax1.plot(depth_results['depth'], depth_results['test_acc'], marker='s', label='Test Accuracy', color='tab:gray', linestyle='--')
ax1.tick_params(axis='y', labelcolor='tab:blue')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)

# Time Plot (Right Y-Axis)
ax2 = ax1.twinx()
ax2.set_ylabel('Training Time (Seconds)', color='black')
ax2.bar(depth_results['depth'], depth_results['train_time'], alpha=0.2, color='tab:green', label='Training Time')
ax2.tick_params(axis='y', labelcolor='tab:green')
ax2.legend(loc='lower right')

plt.title('Depth vs Performance: Accuracy and Time for 1, 2, and 4 Hidden Layers')
plt.show()


**Why and why not deeper is not equal to always better**Overfitting: Across all depths, the training accuracy reached nearly 100%, but test accuracy hovered around 79%. The 4-layer model actually performed slightly worse on the test set (78.7%) than the 2-layer model (79.8%). When a model is too deep, it tends to overfit—meaning it memorizes the training data perfectly but loses the ability to generalize to new, unseen data.
Complexity vs. Data: Text classification on this dataset using simple CountVectorizer features doesn't require a highly complex. Adding more layers just adds unnecessary parameters to optimize, which can make the model harder to train (sometimes leading to vanishing gradients).
Training Time: Interestingly, the 4-layer model converged faster in terms of time here. This can happen because with more parameters, the optimizer might find a local minimum faster and trigger early stopping, or simply because it memorized the data so fast. However, generally speaking, more layers mean more computational cost per epoch.

# Failure Analysis

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer

# 1. Recover the original text for the test set to see actual sentences
text_data = subset_df['text'].values
_, text_test = train_test_split(text_data, test_size=0.2, random_state=42)

# Recreate the 1000-feature test set for the Perceptron
vectorizer_1k = CountVectorizer(max_features=1000)
X_1k = vectorizer_1k.fit_transform(subset_df['text']).toarray()
_, X_test_1k, _, _ = train_test_split(X_1k, y, test_size=0.2, random_state=42)

# 2. Get predictions on the test set
# Perceptron predictions (requires 1000 features)
vp_preds = vp.predict(X_test_1k)

# Neural Network predictions (mlp was trained on the 7453-feature set)
nn_preds = mlp.predict(X_test)

# 3. Identify misclassified examples
vp_errors = (vp_preds != y_test)
nn_errors = (nn_preds != y_test)
both_errors = vp_errors & nn_errors

# 4. Create a DataFrame to view the results clearly
error_df = pd.DataFrame({
    'Text': text_test,
    'True Label': y_test,
    'Perceptron Pred': vp_preds,
    'Neural Net Pred': nn_preds
})

print("--- Error Rate Comparison ---")
print(f"Perceptron Test Error Rate: {np.mean(vp_errors):.2%}")
print(f"Neural Network Test Error Rate: {np.mean(nn_errors):.2%}\n")

print("--- Examples where BOTH models failed ---")
display(error_df[both_errors].head())

print("\n--- Examples where ONLY Perceptron failed (NN got it right) ---")
display(error_df[vp_errors & ~nn_errors].head())




**1. Why did the models fail? Bag-of-Words Limitation:
The primary reason for failure in models is the reliance on a Bag-of-Words (BoW) vectorizer (`CountVectorizer`). BoW entirely ignores **word order, grammar, and context**.
* For instance, the phrases *"not good, just completely bad"* and *"not bad, just completely good"* possess the exact same word counts, resulting in identical BoW vectors despite having opposite sentiments.
* Sarcasm, idioms, and multi-word expressions (like "over the top") are completely lost, making it impossible for the model to capture deep semantic meaning.

**2. Linear Separability & Model Comparison (Perceptron vs. Neural Network)**
* **Perceptron Errors:** The Voted Perceptron is a linear classifier. It attempts to draw a straight line (or flat hyperplane) to separate positive texts from negative texts. If the data is not linearly separable (e.g., sentiments that depend on complex, non-linear combinations of words), the Perceptron will mathematically fail to classify them correctly.
* **Neural Network Errors:** The Neural Network, equipped with hidden layers and non-linear activation functions (ReLU), can learn non-linear decision boundaries. This is why the NN succeeded while the Perceptron failed. However, the NN still fails on many examples because no amount of non-linear capability can recover the contextual information that was erased by the Bag-of-Words representation in the first place.

# Optimization Techniques


*   With /Without Regularization  **Test accuracy doesn't change much using this techniques, indicating the model may not capture enough siginals and may underfitting**

*   Relu VS TanH VS Sigmoid **Relu learns fasters and its gradients don't disapper which avoids the vanishing gradient problems. Relu keeps derivative exactly 1 for any positive number so to avoide the curve becomes flat. Both sigmoid and Tanch sometimes have problems with stopping updating wieghts and the network stops learning when data are verylarge or very small**
*   Learning rate tunning **can effectively reduce training loss**

* Cost function **The two linear cost functions gave almost the same test accuracy, so switching from cross-entropy to hinge loss did not meaningfully improve performance.**



In [ ]:
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import SGDClassifier
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)

print("Starting Part 6: Optimization Techniques Experiments...")

fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. Regularization (With vs Without L2 Penalty)
print("1. Testing Regularization...")
alphas = {'Without': 0.0, 'With (alpha=0.01)': 0.01}
for name, alpha in alphas.items():
    mlp = MLPClassifier(hidden_layer_sizes=(64,), alpha=alpha, max_iter=40, random_state=42)
    mlp.fit(X_train, y_train)
    axes[0, 0].plot(mlp.loss_curve_, label=f'{name} (Test Acc: {mlp.score(X_test, y_test):.3f})')
axes[0, 0].set_title('1. Regularization (L2 Penalty)')
axes[0, 0].set_xlabel('Epochs')
axes[0, 0].set_ylabel('Training Loss')
axes[0, 0].legend()

# 2. Activation Functions (ReLU vs Tanh vs Sigmoid)
print("2. Testing Activations...")
activations = ['relu', 'tanh', 'logistic']
for act in activations:
    mlp = MLPClassifier(hidden_layer_sizes=(64,), activation=act, max_iter=40, random_state=42)
    mlp.fit(X_train, y_train)
    name = 'Sigmoid' if act == 'logistic' else act.capitalize()
    axes[0, 1].plot(mlp.loss_curve_, label=f'{name} (Test Acc: {mlp.score(X_test, y_test):.3f})')
axes[0, 1].set_title('2. Activation Functions')
axes[0, 1].set_xlabel('Epochs')
axes[0, 1].set_ylabel('Training Loss')
axes[0, 1].legend()

# 3. Learning Rate Tuning
print("3. Testing Learning Rates...")
lrs = [0.001, 0.01, 0.1]
for lr in lrs:
    mlp = MLPClassifier(hidden_layer_sizes=(64,), learning_rate_init=lr, max_iter=40, random_state=42)
    mlp.fit(X_train, y_train)
    axes[1, 0].plot(mlp.loss_curve_, label=f'LR={lr} (Test Acc: {mlp.score(X_test, y_test):.3f})')
axes[1, 0].set_title('3. Learning Rate Tuning')
axes[1, 0].set_xlabel('Epochs')
axes[1, 0].set_ylabel('Training Loss')
axes[1, 0].legend()

# 4. Cost Function Tuning (Using SGDClassifier to compare Hinge vs Log Loss)
print("4. Testing Cost Functions...")
losses = {'Log Loss (Cross-Entropy)': 'log_loss', 'Hinge Loss (SVM)': 'hinge'}
for name, loss in losses.items():
    sgd = SGDClassifier(loss=loss, max_iter=40, random_state=42)
    sgd.fit(X_train, y_train)
    test_acc = sgd.score(X_test, y_test)
    axes[1, 1].bar(name, test_acc, label=f'{name} Acc: {test_acc:.3f}')
    axes[1, 1].text(name, test_acc / 2, f'{test_acc:.3f}', ha='center', color='white', weight='bold')

axes[1, 1].set_title('4. Cost Function Tuning (Linear Models)')
axes[1, 1].set_ylabel('Test Accuracy')
axes[1, 1].legend(loc='lower right')

plt.tight_layout()
plt.show()
